# HYGD metadata cleaning and cohort audit

**Public role:** Core preprocessing and cohort-definition notebook.

**Relation to the manuscript:** Supports the HYGD cohort definition and audit metadata.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `01_hygd_metadata_cleaning_and_audit.ipynb`


In [ ]:
import os
# Cell 1: Load project config and raw labels

from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing project config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

HYGD_ROOT = Path(PROJECT_CONFIG["hygd_root"])
HYGD_IMAGES_DIR = Path(PROJECT_CONFIG["hygd_images_dir"])
EXPECTED_HYGD_IMAGES = int(PROJECT_CONFIG["expected_hygd_images"])
EXPECTED_HYGD_PATIENTS = int(PROJECT_CONFIG["expected_hygd_patients"])

LABELS_CSV = HYGD_ROOT / "Labels.csv"

if not LABELS_CSV.exists():
    raise FileNotFoundError(f"Missing Labels.csv: {LABELS_CSV}")

labels_raw = pd.read_csv(LABELS_CSV)

raw_label_audit = pd.DataFrame([
    {
        "file": str(LABELS_CSV),
        "n_rows": labels_raw.shape[0],
        "n_columns": labels_raw.shape[1],
        "columns": ", ".join(labels_raw.columns.astype(str)),
    }
])

raw_label_audit_csv = PATHS["results_01_dataset_audit"] / "notebook01_raw_labels_audit.csv"
raw_label_audit.to_csv(raw_label_audit_csv, index=False)

print(f"Labels rows: {labels_raw.shape[0]}")
print(f"Labels columns: {labels_raw.shape[1]}")
print(f"Columns: {list(labels_raw.columns)}")
print(f"Audit CSV: {raw_label_audit_csv}")

if labels_raw.shape[0] != EXPECTED_HYGD_IMAGES:
    raise ValueError(f"Expected {EXPECTED_HYGD_IMAGES} rows, found {labels_raw.shape[0]}")

print("Status: PASS")

In [ ]:
# Cell 2: Standardize labels and resolve image paths

expected_columns = ["Image Name", "Patient", "Label", "Quality Score"]
missing_columns = [col for col in expected_columns if col not in labels_raw.columns]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(labels_raw.columns)}"
    )

metadata = labels_raw.copy()
metadata = metadata.drop(columns=[col for col in metadata.columns if str(col).startswith("Unnamed")])

metadata = metadata.rename(
    columns={
        "Image Name": "image_name",
        "Patient": "patient_id",
        "Label": "label_raw",
        "Quality Score": "quality_score",
    }
)

metadata["image_name"] = metadata["image_name"].astype(str).str.strip()
metadata["patient_id"] = metadata["patient_id"].astype(str).str.strip()
metadata["label_raw"] = metadata["label_raw"].astype(str).str.strip()
metadata["quality_score"] = pd.to_numeric(metadata["quality_score"], errors="coerce")

label_map = {
    "GON+": 1,
    "GON-": 0,
}

unknown_labels = sorted(set(metadata["label_raw"]) - set(label_map))

if unknown_labels:
    raise ValueError(f"Unknown labels found: {unknown_labels}")

metadata["label"] = metadata["label_raw"].map(label_map).astype(int)

image_files = sorted(
    p for p in HYGD_IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in set(PROJECT_CONFIG["image_extensions"])
)

image_by_name = {p.name: p for p in image_files}
image_by_stem = {}

for p in image_files:
    image_by_stem.setdefault(p.stem, []).append(p)

resolved_paths = []
unresolved_images = []
ambiguous_images = []

for image_name in metadata["image_name"]:
    if image_name in image_by_name:
        resolved_paths.append(image_by_name[image_name])
        continue

    stem = Path(image_name).stem
    stem_matches = image_by_stem.get(stem, [])

    if len(stem_matches) == 1:
        resolved_paths.append(stem_matches[0])
    elif len(stem_matches) == 0:
        resolved_paths.append(None)
        unresolved_images.append(image_name)
    else:
        resolved_paths.append(None)
        ambiguous_images.append(image_name)

metadata["image_path"] = [str(p) if p is not None else "" for p in resolved_paths]
metadata["image_exists"] = metadata["image_path"].apply(lambda x: Path(x).exists() if x else False)

quality_missing = int(metadata["quality_score"].isna().sum())
missing_image_paths = int((~metadata["image_exists"]).sum())
n_patients = metadata["patient_id"].nunique()
label_counts = metadata["label_raw"].value_counts().to_dict()

metadata_audit = pd.DataFrame(
    [
        {"check": "rows", "value": len(metadata), "expected": EXPECTED_HYGD_IMAGES, "passed": len(metadata) == EXPECTED_HYGD_IMAGES},
        {"check": "patients", "value": n_patients, "expected": EXPECTED_HYGD_PATIENTS, "passed": n_patients == EXPECTED_HYGD_PATIENTS},
        {"check": "missing_quality_score", "value": quality_missing, "expected": 0, "passed": quality_missing == 0},
        {"check": "missing_image_paths", "value": missing_image_paths, "expected": 0, "passed": missing_image_paths == 0},
        {"check": "unresolved_images", "value": len(unresolved_images), "expected": 0, "passed": len(unresolved_images) == 0},
        {"check": "ambiguous_images", "value": len(ambiguous_images), "expected": 0, "passed": len(ambiguous_images) == 0},
    ]
)

metadata_audit_csv = PATHS["results_01_dataset_audit"] / "notebook01_metadata_standardization_audit.csv"
metadata_preview_csv = PATHS["results_01_dataset_audit"] / "notebook01_metadata_standardized_preview.csv"

metadata_audit.to_csv(metadata_audit_csv, index=False)
metadata.head(20).to_csv(metadata_preview_csv, index=False)

print(f"Rows: {len(metadata)}")
print(f"Patients: {n_patients}")
print(f"Labels: {label_counts}")
print(f"Missing image paths: {missing_image_paths}")
print(f"Missing quality scores: {quality_missing}")
print(f"Audit CSV: {metadata_audit_csv}")

failed_checks = metadata_audit.loc[~metadata_audit["passed"], "check"].tolist()

if failed_checks:
    raise RuntimeError(f"Failed checks: {failed_checks}")

print("Status: PASS")

In [ ]:
# Cell 3: Patient-level and quality audit

patient_summary = (
    metadata.groupby("patient_id")
    .agg(
        n_images=("image_name", "count"),
        n_positive=("label", "sum"),
        n_negative=("label", lambda x: int((x == 0).sum())),
        n_unique_labels=("label", "nunique"),
        patient_label=("label", "max"),
        quality_mean=("quality_score", "mean"),
        quality_std=("quality_score", "std"),
        quality_min=("quality_score", "min"),
        quality_max=("quality_score", "max"),
    )
    .reset_index()
)

patient_summary["quality_std"] = patient_summary["quality_std"].fillna(0.0)

mixed_label_patients = patient_summary.loc[
    patient_summary["n_unique_labels"] > 1, "patient_id"
].tolist()

class_distribution = (
    metadata.groupby(["label_raw", "label"])
    .agg(
        n_images=("image_name", "count"),
        n_patients=("patient_id", "nunique"),
        quality_mean=("quality_score", "mean"),
        quality_std=("quality_score", "std"),
    )
    .reset_index()
)

quality_summary = metadata["quality_score"].describe().to_frame("quality_score").reset_index()
quality_summary = quality_summary.rename(columns={"index": "metric"})

patient_summary_csv = PATHS["metadata"] / "patient_metadata.csv"
class_distribution_csv = PATHS["results_01_dataset_audit"] / "notebook01_class_distribution.csv"
quality_summary_csv = PATHS["results_01_dataset_audit"] / "notebook01_quality_summary.csv"
patient_summary_results_csv = PATHS["results_01_dataset_audit"] / "notebook01_patient_summary.csv"

patient_summary.to_csv(patient_summary_csv, index=False)
patient_summary.to_csv(patient_summary_results_csv, index=False)
class_distribution.to_csv(class_distribution_csv, index=False)
quality_summary.to_csv(quality_summary_csv, index=False)

print(f"Patients: {len(patient_summary)}")
print(f"Mixed-label patients: {len(mixed_label_patients)}")
print(f"Images per patient: min={patient_summary['n_images'].min()}, max={patient_summary['n_images'].max()}")
print(f"Quality score: mean={metadata['quality_score'].mean():.3f}, std={metadata['quality_score'].std():.3f}")
print(f"Patient metadata: {patient_summary_csv}")

if len(patient_summary) != EXPECTED_HYGD_PATIENTS:
    raise ValueError(f"Expected {EXPECTED_HYGD_PATIENTS} patients, found {len(patient_summary)}")

if mixed_label_patients:
    raise ValueError(f"Mixed-label patients found: {mixed_label_patients[:20]}")

print("Status: PASS")

In [ ]:
# Cell 4: Save clean metadata and dataset summary

metadata_clean = metadata.copy()

metadata_clean = metadata_clean.merge(
    patient_summary[["patient_id", "n_images", "patient_label"]],
    on="patient_id",
    how="left"
)

metadata_clean = metadata_clean.sort_values(
    ["patient_id", "image_name"]
).reset_index(drop=True)

metadata_clean.insert(
    0,
    "image_id",
    [f"HYGD_{idx:04d}" for idx in range(len(metadata_clean))]
)

metadata_clean["image_relative_path"] = metadata_clean["image_path"].apply(
    lambda x: str(Path(x).relative_to(HYGD_ROOT))
)

final_columns = [
    "image_id",
    "image_name",
    "patient_id",
    "label_raw",
    "label",
    "patient_label",
    "quality_score",
    "n_images",
    "image_relative_path",
    "image_path",
    "image_exists",
]

missing_final_columns = [col for col in final_columns if col not in metadata_clean.columns]

if missing_final_columns:
    raise ValueError(f"Missing final columns: {missing_final_columns}")

metadata_clean = metadata_clean[final_columns]

dataset_summary = {
    "dataset_name": "HYGD",
    "n_images": int(len(metadata_clean)),
    "n_patients": int(metadata_clean["patient_id"].nunique()),
    "n_positive_images": int((metadata_clean["label"] == 1).sum()),
    "n_negative_images": int((metadata_clean["label"] == 0).sum()),
    "n_positive_patients": int((patient_summary["patient_label"] == 1).sum()),
    "n_negative_patients": int((patient_summary["patient_label"] == 0).sum()),
    "mixed_label_patients": int((patient_summary["n_unique_labels"] > 1).sum()),
    "quality_mean": float(metadata_clean["quality_score"].mean()),
    "quality_std": float(metadata_clean["quality_score"].std()),
    "images_per_patient_min": int(patient_summary["n_images"].min()),
    "images_per_patient_max": int(patient_summary["n_images"].max()),
}

metadata_clean_csv = PATHS["metadata"] / "metadata_clean.csv"
dataset_summary_json = PATHS["metadata"] / "dataset_summary.json"
metadata_clean_results_csv = PATHS["results_01_dataset_audit"] / "notebook01_metadata_clean.csv"

metadata_clean.to_csv(metadata_clean_csv, index=False)
metadata_clean.to_csv(metadata_clean_results_csv, index=False)

with open(dataset_summary_json, "w", encoding="utf-8") as f:
    json.dump(dataset_summary, f, indent=2)

duplicate_images = int(metadata_clean["image_name"].duplicated().sum())
missing_paths = int((~metadata_clean["image_exists"]).sum())

print(f"Clean metadata rows: {len(metadata_clean)}")
print(f"Patients: {metadata_clean['patient_id'].nunique()}")
print(f"Duplicate image names: {duplicate_images}")
print(f"Missing image paths: {missing_paths}")
print(f"Metadata CSV: {metadata_clean_csv}")
print(f"Dataset summary: {dataset_summary_json}")

if len(metadata_clean) != EXPECTED_HYGD_IMAGES:
    raise ValueError(f"Expected {EXPECTED_HYGD_IMAGES} rows, found {len(metadata_clean)}")

if metadata_clean["patient_id"].nunique() != EXPECTED_HYGD_PATIENTS:
    raise ValueError("Unexpected patient count.")

if duplicate_images > 0:
    raise ValueError("Duplicate image names found.")

if missing_paths > 0:
    raise ValueError("Missing image paths found.")

print("Status: PASS")

In [ ]:
# Cell 5: Final status

required_outputs = {
    "metadata_clean_csv": metadata_clean_csv,
    "patient_summary_csv": patient_summary_csv,
    "dataset_summary_json": dataset_summary_json,
    "raw_label_audit_csv": raw_label_audit_csv,
    "metadata_audit_csv": metadata_audit_csv,
    "class_distribution_csv": class_distribution_csv,
    "quality_summary_csv": quality_summary_csv,
}

output_checks = {
    key: path.exists()
    for key, path in required_outputs.items()
}

content_checks = {
    "n_images": len(metadata_clean) == EXPECTED_HYGD_IMAGES,
    "n_patients": metadata_clean["patient_id"].nunique() == EXPECTED_HYGD_PATIENTS,
    "no_missing_image_paths": int((~metadata_clean["image_exists"]).sum()) == 0,
    "no_missing_quality_scores": int(metadata_clean["quality_score"].isna().sum()) == 0,
    "no_duplicate_image_names": int(metadata_clean["image_name"].duplicated().sum()) == 0,
    "no_mixed_label_patients": int((patient_summary["n_unique_labels"] > 1).sum()) == 0,
}

final_checks = {**output_checks, **content_checks}
failed_checks = [key for key, passed in final_checks.items() if not passed]
status = "PASS" if not failed_checks else "FAIL"

notebook01_status = {
    "notebook": "01_hygd_metadata_cleaning_and_audit",
    "status": status,
    "n_images": int(len(metadata_clean)),
    "n_patients": int(metadata_clean["patient_id"].nunique()),
    "n_positive_images": int((metadata_clean["label"] == 1).sum()),
    "n_negative_images": int((metadata_clean["label"] == 0).sum()),
    "quality_mean": float(metadata_clean["quality_score"].mean()),
    "quality_std": float(metadata_clean["quality_score"].std()),
    "failed_checks": failed_checks,
    "next_notebook": "02_hygd_patient_safe_splits_and_groupkfold.ipynb",
}

status_json = PATHS["results_01_dataset_audit"] / "notebook01_final_status.json"
status_txt = PATHS["results_01_dataset_audit"] / "notebook01_FINAL_STATUS.txt"
status_checks_csv = PATHS["results_01_dataset_audit"] / "notebook01_final_status_checks.csv"

with open(status_json, "w", encoding="utf-8") as f:
    json.dump(notebook01_status, f, indent=2)

with open(status_txt, "w", encoding="utf-8") as f:
    for key, value in notebook01_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": value} for key, value in final_checks.items()]
).to_csv(status_checks_csv, index=False)

print(f"Status: {status}")
print(f"Images: {len(metadata_clean)}")
print(f"Patients: {metadata_clean['patient_id'].nunique()}")
print(f"Labels: {metadata_clean['label_raw'].value_counts().to_dict()}")
print(f"Final status: {status_json}")

if failed_checks:
    raise RuntimeError(f"Notebook 01 failed checks: {failed_checks}")